# CheXpert Model Evaluation

Comprehensive evaluation of the CheXpert multi-label classification model across all 14 pathology labels.

## Methodology

Evaluation is performed on the **CheXpert validation set** (200 frontal chest radiographs with radiologist-adjudicated labels).
We report:
- **AUC-ROC** per label and macro-averaged
- **Calibration** via Expected Calibration Error (ECE) and reliability diagrams
- **Confusion matrices** for discretised predictions (threshold = 0.5)

Labels follow the CheXpert uncertainty handling convention:
`negative → 0`, `uncertain → 0` (U-Ignore), `positive → 1`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.calibration import calibration_curve

plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor':   '#1e293b',
    'axes.edgecolor':   '#334155',
    'axes.labelcolor':  '#cbd5e1',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'text.color':       '#e2e8f0',
    'grid.color':       '#1e293b',
    'grid.linewidth':   0.5,
    'legend.facecolor': '#1e293b',
    'legend.edgecolor': '#334155',
})

LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
    'Lung Opacity', 'No Finding', 'Pleural Effusion',
    'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices',
]
N_LABELS = len(LABELS)
N_SAMPLES = 200

# Reproducible synthetic ground truth + predictions for demonstration.
# In production: replace with actual model outputs.
rng = np.random.default_rng(42)
PREVALENCE = [
    0.29, 0.08, 0.06, 0.18, 0.07, 0.03, 0.05,
    0.35, 0.10, 0.38, 0.04, 0.06, 0.06, 0.44,
]
y_true = rng.binomial(1, np.array(PREVALENCE), size=(N_SAMPLES, N_LABELS)).astype(float)

# Simulate model probabilities: calibrated around true labels + noise
logit_noise = rng.normal(0, 1.2, size=(N_SAMPLES, N_LABELS))
logits = 2.0 * (y_true - 0.5) + logit_noise
y_pred = 1.0 / (1.0 + np.exp(-logits))

print(f'Evaluation set: {N_SAMPLES} studies × {N_LABELS} labels')
print('Ground truth label prevalence:')
for lbl, prev in zip(LABELS, y_true.mean(axis=0)):
    print(f'  {lbl:<30} {prev:.3f}')

## 1. ROC Curves — All 14 Labels

In [ ]:
TEAL   = '#2dd4bf'
COLORS = plt.cm.tab20(np.linspace(0, 1, N_LABELS))

aucs = {}
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.ravel()

for i, label in enumerate(LABELS):
    fpr, tpr, _ = roc_curve(y_true[:, i], y_pred[:, i])
    roc_auc = auc(fpr, tpr)
    aucs[label] = roc_auc

    ax = axes[i]
    ax.plot(fpr, tpr, color=COLORS[i], lw=2)
    ax.plot([0, 1], [0, 1], '--', color='#475569', lw=1)
    ax.set_title(f'{label}\nAUC = {roc_auc:.3f}', fontsize=9, pad=4)
    ax.set_xlabel('FPR', fontsize=8)
    ax.set_ylabel('TPR', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

# Hide spare subplot
axes[-1].set_visible(False)
axes[-2].set_visible(False)

fig.suptitle('ROC Curves — CheXpert 14-Label Validation Set', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/roc_curves.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print(f'Macro-average AUC: {np.mean(list(aucs.values())):.4f}')

## 2. AUC Summary Table

In [ ]:
auc_df = pd.DataFrame({
    'Label':      list(aucs.keys()),
    'AUC':        [round(v, 4) for v in aucs.values()],
    'Prevalence': [round(float(y_true[:, i].mean()), 3) for i, _ in enumerate(LABELS)],
    'N Positive': [int(y_true[:, i].sum()) for i in range(N_LABELS)],
}).sort_values('AUC', ascending=False).reset_index(drop=True)

auc_df.index += 1  # 1-based rank
print('AUC per label (ranked):')
print(auc_df.to_string())
print(f'\nMacro-average AUC: {auc_df.AUC.mean():.4f}')

## 3. Calibration Plots

Reliability diagrams show how well the predicted probabilities align with the empirical positive fraction. A perfectly calibrated model follows the diagonal.

In [ ]:
def expected_calibration_error(y_true_col, y_pred_col, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_pred_col >= lo) & (y_pred_col < hi)
        if mask.sum() == 0:
            continue
        acc  = y_true_col[mask].mean()
        conf = y_pred_col[mask].mean()
        ece += mask.sum() / len(y_pred_col) * abs(acc - conf)
    return ece

fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.ravel()
ece_scores = {}

for i, label in enumerate(LABELS):
    prob_true, prob_pred = calibration_curve(y_true[:, i], y_pred[:, i], n_bins=8)
    ece = expected_calibration_error(y_true[:, i], y_pred[:, i])
    ece_scores[label] = ece

    ax = axes[i]
    ax.plot(prob_pred, prob_true, 's-', color=COLORS[i], lw=1.5, markersize=4)
    ax.plot([0, 1], [0, 1], '--', color='#475569', lw=1)
    ax.fill_between(prob_pred, prob_pred, prob_true, alpha=0.15, color=COLORS[i])
    ax.set_title(f'{label}\nECE = {ece:.3f}', fontsize=9, pad=4)
    ax.set_xlabel('Mean predicted prob.', fontsize=8)
    ax.set_ylabel('Fraction of positives', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])

axes[-1].set_visible(False)
axes[-2].set_visible(False)

fig.suptitle('Calibration (Reliability) Diagrams — All 14 Labels', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/calibration_plots.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print(f'Mean ECE: {np.mean(list(ece_scores.values())):.4f}')

## 4. Confusion Matrices (threshold = 0.5)

In [ ]:
THRESHOLD = 0.5
y_pred_binary = (y_pred >= THRESHOLD).astype(int)

fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.ravel()

for i, label in enumerate(LABELS):
    cm = confusion_matrix(y_true[:, i], y_pred_binary[:, i])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Neg', 'Pos'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(label, fontsize=9)
    axes[i].tick_params(labelsize=8)

axes[-1].set_visible(False)
axes[-2].set_visible(False)

fig.suptitle(f'Confusion Matrices — Threshold = {THRESHOLD}', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.show()

# Per-label classification report summary
print('\nPer-label classification summary (threshold = 0.5):')
rows = []
for i, label in enumerate(LABELS):
    report = classification_report(
        y_true[:, i], y_pred_binary[:, i],
        target_names=['neg', 'pos'], output_dict=True, zero_division=0,
    )
    rows.append({
        'Label':     label,
        'Precision': round(report['pos']['precision'], 3),
        'Recall':    round(report['pos']['recall'], 3),
        'F1':        round(report['pos']['f1-score'], 3),
        'Support':   int(report['pos']['support']),
    })
print(pd.DataFrame(rows).sort_values('F1', ascending=False).to_string(index=False))